# 02 — Delta Training (Peak-Capture)

This notebook trains the RefineNet post-processor on the DF2K dataset
using the peak-capture strategy:
1. Train with constant LR=1e-6
2. Evaluate Set5 every 50 iterations
3. Save best checkpoint
4. Stop after 5 consecutive non-improvements

> **Note**: If you already have a trained checkpoint, skip to
> notebook 03 for evaluation and visualization.

In [ ]:
import sys, os, copy, random
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from src.models.cfsr import CFSR
from src.models.refine_net import RefineNet
from src.models.cfsr_delta import CFSRDelta
from src.data.df2k_dataset import DF2KDataset
from src.metrics.sr_metrics import evaluate_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

In [ ]:
# Paths
BACKBONE_WEIGHTS = '../model_zoo/CFSR_x4.pth'
DATA_DIR = '../datasets/DF2K/HR'
BENCH_DIR = '../datasets/benchmark'
SAVE_DIR = '../checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

# Hyperparameters
SEED = 69
SCALE = 4
HIDDEN_CH = 32
INIT_SCALE = 1e-4
LR = 1e-6
MAX_ITERS = 10_000
EVAL_INTERVAL = 50
PATIENCE = 5
BATCH_SIZE = 16
PATCH_SIZE = 64
GRAD_CLIP = 0.5

# Reproducibility
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

## Load Backbone + Initialize RefineNet

In [ ]:
# Backbone (frozen)
backbone = CFSR(); backbone.merge_all()
sd = torch.load(BACKBONE_WEIGHTS, map_location='cpu', weights_only=True)
if 'params' in sd: sd = sd['params']
backbone.load_state_dict(sd, strict=True)
backbone = backbone.to(device).eval()
print(f'Backbone: {backbone.param_count():,} params (frozen)')

# RefineNet
torch.manual_seed(SEED)
refine = RefineNet(hidden_channels=HIDDEN_CH, init_scale=INIT_SCALE)
print(f'RefineNet: {refine.param_count():,} params (trainable)')

model = CFSRDelta(backbone, refine).to(device)

## Initial Evaluation

In [ ]:
init_result = evaluate_dataset(model, 'Set5', SCALE, BENCH_DIR, str(device))
baseline_psnr = init_result['psnr']
print(f'Starting PSNR: {baseline_psnr:.4f} dB')

## Training Loop (Peak-Capture)

In [ ]:
dataset = DF2KDataset(DATA_DIR, SCALE, PATCH_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

optimizer = optim.Adam(refine.parameters(), lr=LR, betas=(0.9, 0.99))
criterion = nn.L1Loss()

best_psnr = baseline_psnr or 0.0
best_state = copy.deepcopy(refine.state_dict())
best_iter = 0
non_improve = 0
train_log = []

print(f'Training: max {MAX_ITERS} iters, eval every {EVAL_INTERVAL}, patience {PATIENCE}')
print(f'LR={LR}, grad_clip={GRAD_CLIP}\n')

model.train()
it = 0; stopped = False

for lr_b, hr_b in loader:
    if it >= MAX_ITERS or stopped: break
    lr_b, hr_b = lr_b.to(device), hr_b.to(device)
    loss = criterion(model(lr_b), hr_b)
    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(refine.parameters(), GRAD_CLIP)
    optimizer.step(); it += 1

    if it % EVAL_INTERVAL == 0:
        result = evaluate_dataset(model, 'Set5', SCALE, BENCH_DIR, str(device))
        model.train()
        psnr = result['psnr']
        gain = psnr - (baseline_psnr or psnr)
        mark = ''
        if psnr > best_psnr:
            best_psnr, best_iter = psnr, it
            best_state = copy.deepcopy(refine.state_dict())
            mark = ' *** BEST'; non_improve = 0
        else:
            non_improve += 1
        train_log.append((it, psnr, gain))
        print(f'  [iter {it:>5d}] PSNR={psnr:.4f} ({gain:+.4f}){mark}')
        if non_improve >= PATIENCE:
            print(f'\n  [EARLY STOP] Peak at iter {best_iter}'); stopped = True

# Save
save_path = os.path.join(SAVE_DIR, f'refine_best_x{SCALE}.pth')
torch.save({'refine': best_state, 'psnr': float(best_psnr), 'iter': best_iter}, save_path)
print(f'\nBest: PSNR={best_psnr:.4f} at iter {best_iter}')
print(f'Saved: {save_path}')

## Training Curve

In [ ]:
import matplotlib.pyplot as plt

if train_log:
    iters = [x[0] for x in train_log]
    psnrs = [x[1] for x in train_log]
    gains = [x[2] for x in train_log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(iters, psnrs, 'b-o', ms=3); ax1.axhline(baseline_psnr, color='r', ls='--')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Set5 PSNR (dB)'); ax1.set_title('Training PSNR')
    ax1.grid(alpha=0.3); ax1.legend(['Delta', f'Baseline ({baseline_psnr:.4f})'])

    ax2.plot(iters, gains, 'g-o', ms=3); ax2.axhline(0, color='r', ls='--')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Gain (dB)'); ax2.set_title('Gain vs Baseline')
    ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No training log.')